## Text Generation with HuggingFace - GPT2

Small model to launch on your toster: BERT, GPT-2 (500 MB)

Run GPT-2: https://huggingface.co/openai-community/gpt2

Search "huggingface gpt-2 example": https://www.kaggle.com/code/tuckerarrants/text-generation-with-huggingface-gpt2

Beam search:
* SUPER Beam Search Theory: https://huggingface.co/blog/how-to-generate
* Docs for generate(): https://huggingface.co/docs/transformers/en/main_classes/text_generation
* Obtain logits for each step, decode token indices into text: https://discuss.huggingface.co/t/how-can-i-obtain-the-logits-via-model-generate/110636/2

In [1]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

tokenizer = GPT2Tokenizer.from_pretrained("openai-community/gpt2")
model = GPT2LMHeadModel.from_pretrained("openai-community/gpt2")

inputs = tokenizer("Hello, my dog is cute and ", return_tensors="pt")

In [2]:
# Simple output
generation_output = model.generate(**inputs, max_length=20)
print("Output:\n" + 100 * '-')
print(generation_output)
print(tokenizer.decode(generation_output[0]))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Output:
----------------------------------------------------------------------------------------------------
tensor([[15496,    11,   616,  3290,   318, 13779,   290,   220, 17479,    13,
           314,  1101,   407,  1654,   611,   673,   338,   257,   922,  3290]])
Hello, my dog is cute and icky. I'm not sure if she's a good dog


In [3]:
# Return dict
generation_output = model.generate(**inputs, max_length=20, return_dict_in_generate=True, output_scores=True)
print(generation_output.keys())
print(len(generation_output.scores))
print(generation_output.scores[2].shape)
print("Output:\n" + 100 * '-')
tokenizer.decode(generation_output.sequences[0])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


odict_keys(['sequences', 'scores', 'past_key_values'])
12
torch.Size([1, 50257])
Output:
----------------------------------------------------------------------------------------------------


"Hello, my dog is cute and icky. I'm not sure if she's a good dog"

Try beam search.

In [4]:
beam_outputs = model.generate(
    **inputs,
    max_length=20,
    num_beams = 5,
    num_return_sequences = 5,
    return_dict_in_generate=True,
    output_scores=True,
    early_stopping="never"
)
print(beam_outputs.keys())
print("Number of sequences:", len(beam_outputs.sequences))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


odict_keys(['sequences', 'sequences_scores', 'scores', 'beam_indices', 'past_key_values'])
Number of sequences: 5


In [5]:
print("Output:\n" + 100 * '-')
for i in range(len(beam_outputs.sequences)):
    print("Sequence #", i)
    print("Score:", beam_outputs.sequences_scores[i])
    print(tokenizer.decode(beam_outputs.sequences[i]))
    print(100 * '-')

Output:
----------------------------------------------------------------------------------------------------
Sequence # 0
Score: tensor(-1.4429)
Hello, my dog is cute and icky, but I'm not sure if I'm going to
----------------------------------------------------------------------------------------------------
Sequence # 1
Score: tensor(-1.4660)
Hello, my dog is cute and icky, but I'm not sure if he's going to
----------------------------------------------------------------------------------------------------
Sequence # 2
Score: tensor(-1.5870)
Hello, my dog is cute and icky, but I'm not sure if he's a good
----------------------------------------------------------------------------------------------------
Sequence # 3
Score: tensor(-1.6154)
Hello, my dog is cute and icky, but I'm not sure if he's really cute
----------------------------------------------------------------------------------------------------
Sequence # 4
Score: tensor(-1.6290)
Hello, my dog is cute and icky, but I'm not

In [6]:
beam_outputs.sequences

tensor([[15496,    11,   616,  3290,   318, 13779,   290,   220, 17479,    11,
           475,   314,  1101,   407,  1654,   611,   314,  1101,  1016,   284],
        [15496,    11,   616,  3290,   318, 13779,   290,   220, 17479,    11,
           475,   314,  1101,   407,  1654,   611,   339,   338,  1016,   284],
        [15496,    11,   616,  3290,   318, 13779,   290,   220, 17479,    11,
           475,   314,  1101,   407,  1654,   611,   339,   338,   257,   922],
        [15496,    11,   616,  3290,   318, 13779,   290,   220, 17479,    11,
           475,   314,  1101,   407,  1654,   611,   339,   338,  1107, 13779],
        [15496,    11,   616,  3290,   318, 13779,   290,   220, 17479,    11,
           475,   314,  1101,   407,  1654,   611,   673,   338,   257,   922]])

In [6]:
print("steps", len(beam_outputs.scores))
print("(n_seq, vocab_size)", beam_outputs.scores[0].shape)

steps 12
(n_seq, vocab_size) torch.Size([5, 50257])
